In [ ]:
import os
os.environ['LOGS_DIRECTORY'] = '../eval_logs'

import sys
sys.path.insert(0, '..')

In [ ]:
import json
import random

from tqdm.auto import tqdm
from pydantic import BaseModel
from pydantic_ai import Agent

from ingest import index_data
from search_agent import init_agent
from logs import log_interaction_to_file, LOG_DIR

In [ ]:
LOG_DIR.absolute()

In [ ]:
REPO_OWNER = "huggingface"
REPO_NAME = "transformers"

In [ ]:
vindex, embedding_model = index_data(REPO_OWNER, REPO_NAME, folder_filter="docs/source/en")
agent = init_agent(vindex, embedding_model, REPO_OWNER, REPO_NAME)

In [ ]:
question_generation_prompt = """
You are helping to create test questions for an AI agent that answers questions using HuggingFace Transformers
documentation (markdown sections taken from the library docs).
Based on the provided records, generate realistic questions that developers might ask.

The questions should:
- Be natural and varied in style
- Range from simple to complex
- Include both specific technical questions and general library-usage questions

The user message starts with a line BATCH_SIZE=N, then a JSON array of exactly N objects.
Each object has "filename" (repo path string) and "section" (markdown body for that file).
You MUST return exactly N strings in `questions`, in the same order as the array (index i ↔ questions[i]).
Do not merge records; do not return fewer than N questions.
""".strip()

class QuestionsList(BaseModel):
    questions: list[str]

question_generator = Agent(
    name="question_generator",
    instructions=question_generation_prompt,
    model='openai:gpt-4o-mini',
    output_type=QuestionsList
)

In [ ]:
sample = random.sample(vindex.docs, 10)
# Match the prompt: one object per record (filename + body as "section").
records = [{"filename": d["filename"], "section": d["content"]} for d in sample]
n = len(records)
payload = json.dumps(records, ensure_ascii=False)
prompt = (
    f"BATCH_SIZE={n}\n"
    f"The JSON array has exactly {n} objects (indices 0..{n - 1}). "
    f"Return exactly {n} questions in field `questions`, same order.\n\n"
    f"{payload}"
)

In [ ]:
result = await question_generator.run(prompt)
question_list = result.output

In [ ]:
for question in tqdm(question_list.questions):
    print(question)
    result = await agent.run(user_prompt=question)
    print(result.output)
    log_interaction_to_file(agent, result.new_messages(), source='ai-generated')
    print()